# Module 4: Benchmark Dashboard

Visualize and compare all quantization schemes.

**Run benchmarks first:**
```bash
python benchmarks/benchmark_quant.py --quick --schemes fp16 absmax_int8 zeropoint_int8
```

This notebook generates 4 charts:
1. Memory vs Perplexity scatter (Pareto frontier)
2. Generation speed comparison
3. Quantization error heatmaps
4. Perplexity vs bits-per-weight curve

In [ ]:
import sys, json, os
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Load benchmark results
results_path = '../benchmarks/results/quant_results.json'

if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print(f'Loaded {len(results)} benchmark results')
    for r in results:
        print(f"  {r['scheme']:20s}: size={r['model_size_mb']:.0f} MB, ppl={r['perplexity']:.2f}, {r['tokens_per_second']:.1f} tok/s")
else:
    print('No results found. Using example data for visualization.')
    print('Run: python benchmarks/benchmark_quant.py --quick')
    # Example data for demonstration
    results = [
        {'scheme': 'FP16', 'model_size_mb': 2200, 'perplexity': 7.8, 'tokens_per_second': 25, 'ttft_ms': 120, 'peak_memory_mb': 2300},
        {'scheme': 'absmax_INT8', 'model_size_mb': 2200, 'perplexity': 7.95, 'tokens_per_second': 28, 'ttft_ms': 110, 'peak_memory_mb': 2300},
        {'scheme': 'zeropoint_INT8', 'model_size_mb': 2200, 'perplexity': 7.9, 'tokens_per_second': 27, 'ttft_ms': 112, 'peak_memory_mb': 2300},
        {'scheme': 'GPTQ_INT4', 'model_size_mb': 1100, 'perplexity': 8.3, 'tokens_per_second': 32, 'ttft_ms': 90, 'peak_memory_mb': 1200},
    ]

In [ ]:
# Chart 1: Memory vs Perplexity Scatter (Pareto Frontier)
fig, ax = plt.subplots(figsize=(10, 7))

colors = plt.cm.viridis(np.linspace(0, 0.9, len(results)))

for i, r in enumerate(results):
    ax.scatter(r['model_size_mb'], r['perplexity'],
               s=200, color=colors[i], zorder=5, edgecolors='black', linewidth=1)
    ax.annotate(r['scheme'],
                (r['model_size_mb'], r['perplexity']),
                textcoords='offset points', xytext=(10, 5),
                fontsize=11, fontweight='bold')

# Draw Pareto frontier
sorted_results = sorted(results, key=lambda x: x['model_size_mb'])
pareto_x = [r['model_size_mb'] for r in sorted_results]
pareto_y = [r['perplexity'] for r in sorted_results]
ax.plot(pareto_x, pareto_y, 'k--', alpha=0.3, label='Pareto frontier')

ax.set_xlabel('Model Size (MB)  ← smaller is better', fontsize=12)
ax.set_ylabel('Perplexity (WikiText-2)  ← lower is better', fontsize=12)
ax.set_title('Memory vs Quality Tradeoff\n(bottom-left = best: small AND high quality)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Add annotation explaining the ideal direction
ax.annotate('← Better →', xy=(0.02, 0.02), xycoords='axes fraction',
            fontsize=10, color='green', style='italic')

plt.tight_layout()
plt.savefig('../benchmarks/results/chart1_memory_vs_perplexity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 1 saved!')

In [ ]:
# Chart 2: Generation Speed Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

schemes = [r['scheme'] for r in results]
tps = [r['tokens_per_second'] for r in results]
ttft = [r['ttft_ms'] for r in results]
sizes = [r['model_size_mb'] for r in results]

# Normalize sizes for color mapping
size_norm = [(s - min(sizes)) / (max(sizes) - min(sizes) + 1) for s in sizes]
bar_colors = plt.cm.RdYlGn_r(size_norm)  # red=large model, green=small model

bars1 = ax1.bar(schemes, tps, color=bar_colors, edgecolor='black', linewidth=0.5)
ax1.set_title('Generation Speed (tokens/second)\nHigher = faster', fontsize=12)
ax1.set_ylabel('Tokens per second')
ax1.tick_params(axis='x', rotation=15)

# Add value labels on bars
for bar, val in zip(bars1, tps):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{val:.1f}', ha='center', va='bottom', fontweight='bold')

bars2 = ax2.bar(schemes, ttft, color=bar_colors, edgecolor='black', linewidth=0.5)
ax2.set_title('Time to First Token (ms)\nLower = faster', fontsize=12)
ax2.set_ylabel('Milliseconds')
ax2.tick_params(axis='x', rotation=15)

for bar, val in zip(bars2, ttft):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'{val:.0f}ms', ha='center', va='bottom', fontweight='bold')

# Legend for color coding
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=plt.cm.RdYlGn_r(0), label='Larger model'),
                   Patch(facecolor=plt.cm.RdYlGn_r(1), label='Smaller model')]
ax1.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('../benchmarks/results/chart2_speed.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 3: Quantization Error Heatmaps
# Show error patterns for different quantization schemes on one attention layer
from src.quantization.absmax import absmax_quantize, absmax_dequantize
from src.quantization.gptq import gptq_quantize_layer
from src.quantization.gptq_utils import compute_hessian, cholesky_inverse

torch.manual_seed(42)
# Simulated attention weight matrix
W = torch.randn(64, 64) * 0.1
X_calib = torch.randn(64, 500) * 0.5

# INT8 absmax
q8, s8 = absmax_quantize(W, bits=8)
W_int8 = absmax_dequantize(q8, s8)
err_int8 = (W - W_int8).abs()

# Naive INT4
q4, s4 = absmax_quantize(W, bits=4)
W_int4 = absmax_dequantize(q4, s4)
err_int4 = (W - W_int4).abs()

# GPTQ INT4
H = compute_hessian(X_calib)
H_inv = cholesky_inverse(H)
W_gptq_int, scales_gptq, _ = gptq_quantize_layer(W, H_inv, bits=4)
W_gptq = W_gptq_int.float() * scales_gptq.unsqueeze(0)
err_gptq = (W - W_gptq).abs()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
vmax = err_int4.max().item()

for ax, err, title in zip(axes,
    [err_int8, err_int4, err_gptq],
    ['Absmax INT8 Error', 'Naive INT4 Error', 'GPTQ INT4 Error']):
    im = ax.imshow(err[:32, :32].numpy(), cmap='hot', vmin=0, vmax=vmax)
    ax.set_title(f'{title}\nmean={err.mean():.5f}, max={err.max():.5f}')
    ax.set_xlabel('Input dimension')
    ax.set_ylabel('Output dimension')
    plt.colorbar(im, ax=ax)

plt.suptitle('Quantization Error Heatmaps (32×32 slice)\nDarker = Less Error = Better', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../benchmarks/results/chart3_error_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'GPTQ improvement over naive INT4: {err_int4.mean()/err_gptq.mean():.2f}x less error')

In [ ]:
# Chart 4: Perplexity vs Bits-Per-Weight Curve
# Shows the quality cliff below 4 bits

# Typical values for TinyLlama (fill in with your measured values)
bpw_data = [
    {'bits': 2.0, 'ppl': 35.0, 'label': '2-bit (experimental)'},
    {'bits': 3.0, 'ppl': 14.0, 'label': '3-bit'},
    {'bits': 4.0, 'ppl': 9.5,  'label': 'Naive INT4'},
    {'bits': 4.5, 'ppl': 8.3,  'label': 'GPTQ INT4 / Q4_K'},
    {'bits': 8.0, 'ppl': 7.95, 'label': 'Absmax INT8'},
    {'bits': 16.0, 'ppl': 7.8, 'label': 'FP16'},
    {'bits': 32.0, 'ppl': 7.8, 'label': 'FP32'},
]

# Overlay with your actual results
bits_map = {'FP16': 16, 'fp16': 16, 'absmax_INT8': 8, 'zeropoint_INT8': 8,
            'absmax_int8': 8, 'zeropoint_int8': 8, 'GPTQ_INT4': 4.5, 'gptq_int4': 4.5}

fig, ax = plt.subplots(figsize=(10, 6))

# Plot the theoretical curve
x = [d['bits'] for d in bpw_data]
y = [d['ppl'] for d in bpw_data]
ax.plot(x, y, 'b-o', linewidth=2, markersize=8, alpha=0.6, label='Typical values')

for d in bpw_data:
    ax.annotate(d['label'], (d['bits'], d['ppl']), 
                textcoords='offset points', xytext=(5, 5), fontsize=9, color='blue', alpha=0.7)

# Overlay actual measurements if available
for r in results:
    bits = bits_map.get(r['scheme'])
    if bits and r['perplexity'] > 0:
        ax.scatter(bits, r['perplexity'], s=300, color='red', zorder=10, 
                   marker='*', edgecolors='darkred', linewidth=1)
        ax.annotate(f"Your {r['scheme']}\n{r['perplexity']:.2f}",
                    (bits, r['perplexity']),
                    textcoords='offset points', xytext=(-60, 10),
                    fontsize=9, color='red', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

# Mark the quality cliff
ax.axvspan(0, 4, alpha=0.1, color='red', label='Quality cliff zone (< 4 bit)')
ax.axvline(x=4, color='orange', linestyle='--', linewidth=2, alpha=0.7)
ax.text(4.1, max(y)*0.9, '← Quality cliff\nat 4-bit', color='orange', fontsize=10)

ax.set_xlabel('Bits per weight', fontsize=12)
ax.set_ylabel('Perplexity (lower = better)', fontsize=12)
ax.set_title('Quality vs Compression Tradeoff\n(* = your measured results)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0, 34)

plt.tight_layout()
plt.savefig('../benchmarks/results/chart4_perplexity_vs_bits.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final summary table
print('='*70)
print('BENCHMARK SUMMARY')
print('='*70)
print(f'{"Scheme":<20} {"Size(MB)":>10} {"Perplexity":>12} {"Tok/s":>8} {"TTFT(ms)":>10}')
print('-'*70)
for r in results:
    print(f"{r['scheme']:<20} {r['model_size_mb']:>10.0f} {r['perplexity']:>12.2f} "
          f"{r['tokens_per_second']:>8.1f} {r['ttft_ms']:>10.1f}")
print('='*70)
print()
print('Charts saved to benchmarks/results/')
print('For detailed interpretation, read docs/results_analysis.md')